# Potencial Talents

This apparently requires a unsupervised ranking

## Importing Dependencies

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, average_precision_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.compose import make_column_transformer

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print('Modules Imported')

Modules Imported


## Import Data

In [2]:
data_path = 'potential-talents.xlsx'

In [3]:
data = pd.read_excel(data_path)

data

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,NaN
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,NaN
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN
...,...,...,...,...,...
99,100,Aspiring Human Resources Manager | Graduating ...,"Cape Girardeau, Missouri",103,NaN
100,101,Human Resources Generalist at Loparex,"Raleigh-Durham, North Carolina Area",500+,NaN
101,102,Business Intelligence and Analytics at Travelers,Greater New York City Area,49,NaN
102,103,Always set them up for Success,Greater Los Angeles Area,500+,NaN


## Exploring Dataset

**X:**

- `id`: unique identifier for candidate (numeric)
- `job_title`: job title for candidate (text)
- `location`: geographical location for candidate (text)
- `connections`: number of connections candidate has, 500+ means over 500 (text)

**Y:**

- `fit`: how fit the candidate is for the role? (numeric, probability between 0-1)

In [4]:
keywords = ['Aspiring human resources','seeking human resources'] 

The words that will be used for semantic search

In [5]:
job_titles = data['job_title']

job_titles

0      2019 C.T. Bauer College of Business Graduate (...
1      Native English Teacher at EPIK (English Progra...
2                  Aspiring Human Resources Professional
3                 People Development Coordinator at Ryan
4        Advisory Board Member at Celal Bayar University
                             ...                        
99     Aspiring Human Resources Manager | Graduating ...
100                Human Resources Generalist at Loparex
101     Business Intelligence and Analytics at Travelers
102                       Always set them up for Success
103     Director Of Administration at Excellence Logging
Name: job_title, Length: 104, dtype: object

Fetching job titles from the dataframe

In [6]:
vectorizer = TfidfVectorizer()

# Creating matrix based on the vectorization of the job titles
job_titles_matrix = vectorizer.fit_transform(job_titles)

# Creating matrix based on the vectorization of the keywords
keywords_matrix = vectorizer.transform(keywords)

In [7]:
# Geting a matrix with the similarity score from comparing the job titles with the keywords
resulting_matrix = cosine_similarity(job_titles_matrix, keywords_matrix)

# Getting the sum score of all tokens from each candidate
#resulting_matrix = resulting_matrix.sum(axis=1)

# Getting the mean score of all tokens from each candidate
#resulting_matrix = resulting_matrix.mean(axis=1)

# Getting the max score of all tokens from each candidate
resulting_matrix = resulting_matrix.max(axis=1)

In [8]:
# Adding the score data to the fit column
data = data.assign(fit=resulting_matrix)

data

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,0.265312
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,0.000000
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,0.000000
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,0.000000
...,...,...,...,...,...
99,100,Aspiring Human Resources Manager | Graduating ...,"Cape Girardeau, Missouri",103,0.303599
100,101,Human Resources Generalist at Loparex,"Raleigh-Durham, North Carolina Area",500+,0.242570
101,102,Business Intelligence and Analytics at Travelers,Greater New York City Area,49,0.000000
102,103,Always set them up for Success,Greater Los Angeles Area,500+,0.000000


In [9]:
data.head()

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,0.265312
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,0.000000
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,0.000000
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,0.000000


In [10]:
sorted_fit_data = data.sort_values(by=['fit'], ascending=False)

In [11]:
sorted_fit_data.head(10)

,id,job_title,location,connection,fit
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
32,33,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
16,17,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
20,21,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
57,58,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
96,97,Aspiring Human Resources Professional,"Kokomo, Indiana Area",71,0.753591
45,46,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.753591
23,24,Aspiring Human Resources Specialist,Greater New York City Area,1,0.695679
59,60,Aspiring Human Resources Specialist,Greater New York City Area,1,0.695679
5,6,Aspiring Human Resources Specialist,Greater New York City Area,1,0.695679


## Transforming Features

In [21]:
selected_features = sorted_fit_data[['location','connection','fit']].copy()

In [22]:
selected_features['connection'] = selected_features['connection'].map(lambda x: 500 if x == "500+ " else pd.to_numeric(x))

In [23]:
selected_features.head(10)

,location,connection,fit
2,"Raleigh-Durham, North Carolina Area",44,0.753591
32,"Raleigh-Durham, North Carolina Area",44,0.753591
16,"Raleigh-Durham, North Carolina Area",44,0.753591
20,"Raleigh-Durham, North Carolina Area",44,0.753591
57,"Raleigh-Durham, North Carolina Area",44,0.753591
96,"Kokomo, Indiana Area",71,0.753591
45,"Raleigh-Durham, North Carolina Area",44,0.753591
23,Greater New York City Area,1,0.695679
59,Greater New York City Area,1,0.695679
5,Greater New York City Area,1,0.695679


In [24]:
ohe = OneHotEncoder(sparse_output=False).set_output(transform='pandas')

ohe_transformation = ohe.fit_transform(selected_features[['location']])

selected_features = pd.concat([selected_features, ohe_transformation], axis=1).drop(columns='location')

In [25]:
preprocessor = make_column_transformer(
    (StandardScaler(), ['connection', 'fit']),          
    remainder='passthrough'                              
)

X = preprocessor.fit_transform(selected_features)

In [28]:
# The 7th candidate
role_idx = 6  
role_vector = X[role_idx].reshape(1, -1)

selected_features.iloc[role_idx]

connection                                      44.0
fit                                              1.0
location_Amerika Birleşik Devletleri             0.0
location_Atlanta, Georgia                        0.0
location_Austin, Texas Area                      0.0
location_Baltimore, Maryland                     0.0
location_Baton Rouge, Louisiana Area             0.0
location_Bridgewater, Massachusetts              0.0
location_Cape Girardeau, Missouri                0.0
location_Chattanooga, Tennessee Area             0.0
location_Chicago, Illinois                       0.0
location_Dallas/Fort Worth Area                  0.0
location_Denton, Texas                           0.0
location_Gaithersburg, Maryland                  0.0
location_Greater Atlanta Area                    0.0
location_Greater Boston Area                     0.0
location_Greater Chicago Area                    0.0
location_Greater Grand Rapids, Michigan Area     0.0
location_Greater Los Angeles Area             

In [27]:
selected_features['fit'] = cosine_similarity(X, role_vector).flatten()

In [19]:
selected_features = selected_features.sort_values('fit', ascending=False)

In [20]:
selected_features.head(10)

,connection,fit,location_Amerika Birleşik Devletleri,"location_Atlanta, Georgia","location_Austin, Texas Area","location_Baltimore, Maryland","location_Baton Rouge, Louisiana Area","location_Bridgewater, Massachusetts","location_Cape Girardeau, Missouri","location_Chattanooga, Tennessee Area",...,"location_Milpitas, California","location_Monroe, Louisiana Area","location_Myrtle Beach, South Carolina Area","location_New York, New York","location_Raleigh-Durham, North Carolina Area",location_San Francisco Bay Area,"location_San Jose, California","location_Torrance, California","location_Virginia Beach, Virginia","location_İzmir, Türkiye"
2,44,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
32,44,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
16,44,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
20,44,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
57,44,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
45,44,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
96,71,0.834543,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23,1,0.825223,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
59,1,0.825223,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1,0.825223,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
